In [6]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [7]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [8]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [10]:
# TODO: Read image data, feed into Claude


with open("images/prop7.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []
add_user_message(messages, [
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_bytes,
        },
    },
    {
        "type": "text",
        "text": prompt,
    },
])

chat(messages)

Message(id='msg_01UAE6dxJZUju5Fyj7rfg5PY', container=None, content=[TextBlock(citations=None, text="# Satellite Image Fire Risk Analysis\n\n**1. Residence Identification:**\nThe primary residence is a multi-section structure with gray/light-colored roofing located in the center of the image, featuring an irregular footprint with what appears to be multiple wings or sections, and is completely surrounded by dense forest vegetation with no visible driveway or clear access roads in this view.\n\n**2. Tree Overhang Analysis:**\nMultiple trees have canopies that extend directly over significant portions of the residence's roof, with an estimated 50-75% of the total roof area covered by overhanging branches, particularly dense over the northern and eastern sections of the structure.\n\n**3. Fire Risk Assessment:**\nThe overhanging trees create multiple ember catch points across the roof surface, establish direct fuel continuity between the surrounding forest canopy and the structure, and the

In [14]:
#PDF

with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")
    
    messages = []
    add_user_message(messages, [
        {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes,
        },
    },
    {
        "type": "text",
        "text": "Please summarize the key points from this document in 5 bullet points.",
    },
    ])
    
    response = chat(messages)
    print(text_from_message(response))

# Key Points from the Earth Wikipedia Article

• **Earth's Unique Characteristics**: Earth is the third planet from the Sun and the only known astronomical object to harbor life. It is an ocean world with 70.8% of its surface covered by water, making it unique in sustaining liquid surface water in the Solar System.

• **Physical Properties**: Earth is the densest and largest of the four rocky planets, with a mean radius of 6,371 km. It completes one orbit around the Sun in about 365.25 days and rotates on its tilted axis in approximately 23 hours and 56 minutes, producing seasons.

• **Atmosphere and Climate**: Earth has a dynamic atmosphere composed primarily of nitrogen (78%) and oxygen (21%), with water vapor acting as a greenhouse gas. The current average surface temperature is 14.76°C (58.57°F), maintained by the greenhouse effect.

• **Formation and Age**: Earth formed approximately 4.54 billion years ago from gas and dust in the early Solar System. The Moon likely formed from ma